In [1]:
import pandas as pd

raw_df =  pd.read_csv(
    "crunchbase_investments.csv",
    encoding="latin-1",
    engine="python"
)

In [2]:
region_df = raw_df.groupby(["company_region", "company_state_code"]).agg(
    deal_count=("company_name", "count"),
    total_funding=("raised_amount_usd", "sum"),
    avg_deal_size=("raised_amount_usd", "mean")
).reset_index()

In [3]:
region_df["emerging_hub_score"] = (
    region_df["deal_count"] / region_df["avg_deal_size"]
)

emerging_hubs = region_df.sort_values(
    ["deal_count", "avg_deal_size"],
    ascending=[False, True]
)

In [4]:
#data audit
missing_rates = raw_df.isnull().mean().sort_values(ascending=False) * 100
print(missing_rates)

investor_category_code    95.379232
investor_state_code       31.793077
investor_city             23.605069
investor_country_code     22.699073
raised_amount_usd          6.807263
company_category_code      1.216191
company_city               1.008133
company_state_code         0.930584
funded_quarter             0.005674
funded_month               0.005674
funded_year                0.005674
funded_at                  0.005674
funding_round_type         0.005674
investor_permalink         0.003783
investor_region            0.003783
investor_name              0.003783
company_country_code       0.001891
company_name               0.001891
company_permalink          0.001891
company_region             0.001891
dtype: float64


In [5]:
raw_df.isnull().sum().sort_values(ascending=False)

investor_category_code    50427
investor_state_code       16809
investor_city             12480
investor_country_code     12001
raised_amount_usd          3599
company_category_code       643
company_city                533
company_state_code          492
funded_quarter                3
funded_month                  3
funded_year                   3
funded_at                     3
funding_round_type            3
investor_permalink            2
investor_region               2
investor_name                 2
company_country_code          1
company_name                  1
company_permalink             1
company_region                1
dtype: int64

In [6]:
#is data MCAR or biased?
raw_df["missing_amount"] = raw_df["raised_amount_usd"].isnull()
missing_by_round = raw_df.groupby("funding_round_type")["missing_amount"].mean().sort_values(ascending=False)
print(missing_by_round)

funding_round_type
angel             0.200133
crowdfunding      0.200000
private-equity    0.126050
post-ipo          0.121212
venture           0.099361
series-a          0.040250
other             0.031120
series-b          0.018649
series-c+         0.009752
Name: missing_amount, dtype: float64


In [7]:
#check by year for bias
raw_df["funded_at"] = pd.to_datetime(raw_df["funded_at"], errors="coerce")
raw_df["year"] = raw_df["funded_at"].dt.year

missing_by_year = raw_df.groupby("year")["missing_amount"].mean()

print(missing_by_year)

year
1987.0    0.500000
1995.0    0.000000
1996.0    0.500000
1997.0    0.000000
1998.0    0.181818
1999.0    0.073684
2000.0    0.034091
2001.0    0.094828
2002.0    0.103448
2003.0    0.046512
2004.0    0.060827
2005.0    0.024415
2006.0    0.032583
2007.0    0.055219
2008.0    0.059348
2009.0    0.075584
2010.0    0.063701
2011.0    0.085483
2012.0    0.080930
2013.0    0.083064
Name: missing_amount, dtype: float64


In [8]:
#check for duplicates
duplicates = raw_df.duplicated(
    subset=["company_name", "investor_name", "funded_at"]
).sum()

print("Duplicate records:", duplicates)
raw_df[raw_df.duplicated(subset=["company_name", "investor_name", "funded_at"], keep=False)]

Duplicate records: 185


,company_permalink,company_name,company_category_code,company_country_code,company_state_code,company_region,company_city,investor_permalink,investor_name,investor_category_code,...,investor_region,investor_city,funding_round_type,funded_at,funded_month,funded_quarter,funded_year,raised_amount_usd,missing_amount,year
67,/company/row44,Row44,network_hosting,USA,CA,Los Angeles,Westlake Village,/company/advanced-inflight-alliance,Advanced Inflight Alliance,NaN,...,unknown,NaN,venture,2012-06-11,2012-06,2012-Q2,2012.0,45000000.0,False,2012.0
68,/company/row44,Row44,network_hosting,USA,CA,Los Angeles,Westlake Village,/company/advanced-inflight-alliance,Advanced Inflight Alliance,NaN,...,unknown,NaN,venture,2012-06-11,2012-06,2012-Q2,2012.0,45000000.0,False,2012.0
1859,/company/massachusetts-clean-energy-center,Massachusetts Clean Energy Center,cleantech,USA,MA,Boston,Boston,/company/north-shore-innoventures,North Shore InnoVentures,biotech,...,Boston,Beverly,angel,2013-09-05,2013-09,2013-Q3,2013.0,300000.0,False,2013.0
1860,/company/massachusetts-clean-energy-center,Massachusetts Clean Energy Center,cleantech,USA,MA,Boston,Boston,/company/north-shore-innoventures,North Shore InnoVentures,biotech,...,Boston,Beverly,angel,2013-09-05,2013-09,2013-Q3,2013.0,300000.0,False,2013.0
2123,/company/neumitra,Neumitra,hardware,USA,MA,Boston,Boston,/company/rock-health,Rock Health,health,...,SF Bay,San Francisco,angel,2012-06-01,2012-06,2012-Q2,2012.0,NaN,True,2012.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52063,/company/pinkdingo,Pinkdingo,web,USA,NY,New York,New York,/person/sunil-sani,Sunil Sani,NaN,...,unknown,NaN,angel,2011-10-12,2011-10,2011-Q4,2011.0,1400000.0,False,2011.0
52529,/company/spreadknowledge,Spreadknowledge,sports,USA,PA,Philadelphia,Philadelphia,/person/undisclosed-party,Undisclosed Party,NaN,...,unknown,NaN,angel,2012-01-01,2012-01,2012-Q1,2012.0,200000.0,False,2012.0
52530,/company/spreadknowledge,Spreadknowledge,sports,USA,PA,Philadelphia,Philadelphia,/person/undisclosed-party,Undisclosed Party,NaN,...,unknown,NaN,angel,2012-01-01,2012-01,2012-Q1,2012.0,200000.0,False,2012.0
52798,/company/flotype,Flotype,enterprise,USA,CA,SF Bay,Berkeley,/person/yuri-milner,Yuri Milner,NaN,...,unknown,NaN,angel,2012-01-05,2012-01,2012-Q1,2012.0,1400000.0,False,2012.0
